# 08 — Credit Scoring Mini Project

This notebook defines the Module 1 mini project: designing a production-grade MLOps platform for credit scoring.

## Learning objectives

By the end of this notebook, you will be able to:

- Design a target MLOps architecture for credit scoring.
- Define data sources, features, training, deployment, monitoring, and governance controls.
- Build a simplified credit scoring dataset.
- Train a baseline model.
- Produce a mini project architecture and governance checklist.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, classification_report

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

## 1. Business Context

A retail bank wants to deploy a machine learning model to support credit application decisions.

Business objectives:

- Reduce default losses.
- Maintain acceptable approval rates.
- Improve decision speed.
- Ensure model auditability.
- Provide explainable decisions.

## 2. Target Architecture

In [ ]:
architecture = pd.DataFrame({
    "layer": [
        "Data Sources",
        "Data Pipelines",
        "Feature Store",
        "Training Pipeline",
        "Experiment Tracking",
        "Model Registry",
        "Deployment",
        "Monitoring",
        "Responsible AI",
        "Governance"
    ],
    "credit_scoring_component": [
        "Customer, loan, bureau, payment, macroeconomic data",
        "Validation, cleaning, integration, dataset publication",
        "Debt-to-income, utilization, inquiries, missed payments",
        "Automated model training and evaluation",
        "MLflow run tracking",
        "Approved model versions",
        "Batch scoring or real-time API",
        "AUC, drift, latency, approval rate, default rate",
        "Fairness, SHAP explanations, bias assessment",
        "Documentation, approval workflow, audit trail"
    ]
})
architecture

## 3. Simulated Credit Scoring Dataset

In [ ]:
n = 4000

df = pd.DataFrame({
    "age": np.random.randint(21, 70, n),
    "income": np.random.normal(55000, 18000, n).clip(15000, 180000),
    "loan_amount": np.random.normal(18000, 7000, n).clip(1000, 70000),
    "employment_years": np.random.exponential(6, n).clip(0, 40),
    "credit_utilization": np.random.beta(2.2, 4.5, n),
    "missed_payments": np.random.poisson(0.5, n),
    "recent_inquiries": np.random.poisson(1.1, n)
})

df["loan_to_income"] = df["loan_amount"] / df["income"]
df["has_missed_payment"] = (df["missed_payments"] > 0).astype(int)

logit = (
    -3.2
    + 3.0 * df["credit_utilization"]
    + 0.5 * df["missed_payments"]
    + 0.25 * df["recent_inquiries"]
    + 2.0 * df["loan_to_income"]
    - 0.02 * df["employment_years"]
)

prob_default = 1 / (1 + np.exp(-logit))
df["default"] = np.random.binomial(1, prob_default)

df.head()

In [ ]:
df["default"].mean()

## 4. Baseline Model Training

In [ ]:
features = [
    "age",
    "income",
    "loan_amount",
    "employment_years",
    "credit_utilization",
    "missed_payments",
    "recent_inquiries",
    "loan_to_income",
    "has_missed_payment"
]

X = df[features]
y = df["default"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.3,
    random_state=RANDOM_SEED,
    stratify=y
)

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

pred_proba = model.predict_proba(X_test)[:, 1]
pred_class = (pred_proba >= 0.5).astype(int)

auc = roc_auc_score(y_test, pred_proba)
auc

In [ ]:
print(classification_report(y_test, pred_class))

## 5. Model Card Draft

In [ ]:
model_card = {
    "model_name": "credit_scoring_baseline",
    "model_type": "LogisticRegression",
    "business_owner": "Credit Risk Department",
    "dataset_version": "synthetic_credit_v1",
    "training_date": "YYYY-MM-DD",
    "auc": round(float(auc), 4),
    "status": "candidate",
    "intended_use": "Credit application risk assessment",
    "limitations": "Synthetic educational dataset; not suitable for production use"
}

model_card

## 6. Monitoring Plan

In [ ]:
monitoring_plan = pd.DataFrame({
    "monitoring_area": [
        "Operational",
        "Data Quality",
        "Data Drift",
        "Model Performance",
        "Business KPI",
        "Responsible AI"
    ],
    "metric": [
        "Latency and error rate",
        "Missing values and schema violations",
        "Feature distribution shift",
        "ROC-AUC, recall, calibration",
        "Default rate and approval rate",
        "Group disparity and explainability"
    ],
    "action_if_alert": [
        "Investigate API or infrastructure",
        "Block scoring or trigger data incident",
        "Investigate population change",
        "Trigger model review or retraining",
        "Escalate to business owner",
        "Review fairness and governance controls"
    ]
})

monitoring_plan

## 7. Simulated Drift Scenario

In [ ]:
production = df.sample(1000, random_state=RANDOM_SEED).copy()
production["credit_utilization"] = (production["credit_utilization"] + 0.25).clip(0, 1)
production["loan_to_income"] = production["loan_amount"] / production["income"]
production["has_missed_payment"] = (production["missed_payments"] > 0).astype(int)

drift_summary = pd.DataFrame({
    "feature": ["credit_utilization", "loan_to_income"],
    "training_mean": [df["credit_utilization"].mean(), df["loan_to_income"].mean()],
    "production_mean": [production["credit_utilization"].mean(), production["loan_to_income"].mean()]
})

drift_summary

In [ ]:
plt.figure(figsize=(8, 4))
plt.hist(df["credit_utilization"], bins=30, alpha=0.6, label="Training")
plt.hist(production["credit_utilization"], bins=30, alpha=0.6, label="Production")
plt.title("Credit Utilization Drift")
plt.xlabel("Credit utilization")
plt.ylabel("Frequency")
plt.legend()
plt.tight_layout()
plt.show()

## 8. Governance Checklist

In [ ]:
governance_checklist = pd.DataFrame({
    "control": [
        "Dataset version retained",
        "Training code version retained",
        "Hyperparameters logged",
        "Validation report produced",
        "Approval workflow completed",
        "Monitoring dashboard configured",
        "Responsible AI assessment completed",
        "Rollback plan documented"
    ],
    "status": [
        "To be completed",
        "To be completed",
        "To be completed",
        "To be completed",
        "To be completed",
        "To be completed",
        "To be completed",
        "To be completed"
    ]
})

governance_checklist

## 9. Final Project Deliverables

In [ ]:
deliverables = pd.DataFrame({
    "deliverable": [
        "Architecture diagram",
        "Technology selection report",
        "Model card",
        "Monitoring plan",
        "Governance checklist",
        "RACI matrix",
        "Risk assessment"
    ],
    "expected_content": [
        "End-to-end MLOps components and data flows",
        "Justification of selected tools",
        "Purpose, data, model, metrics, limitations",
        "Metrics, thresholds, alert actions",
        "Auditability and compliance controls",
        "Roles and responsibilities",
        "Key risks and mitigations"
    ]
})

deliverables

## 10. Exercises

### Exercise 1 — Architecture Design

Complete an architecture for the credit scoring platform using the layers shown above.

### Exercise 2 — Tool Selection

Choose one tool for each layer and justify your choice.

### Exercise 3 — Governance

Complete the governance checklist with ownership and evidence.

### Exercise 4 — Monitoring

Define thresholds for:

- AUC degradation.
- Missing values.
- Feature drift.
- API latency.
- Approval rate.

### Exercise 5 — Responsible AI

Define two fairness metrics that could be monitored for credit scoring.

## Key Takeaways

- The mini project connects all concepts introduced in Module 1.
- A credit scoring platform requires data, model, deployment, monitoring, governance, and Responsible AI layers.
- Model performance alone is not sufficient for production readiness.
- Auditability and monitoring must be designed from the beginning.
- This project will be progressively implemented in later modules.